## 03 - Silver Clean and Export

**Layer:** Silver (cleaned data)

### Goal
Standardize the merged dataset, add derived columns and outlier flags,
create a surrogate `listing_id` (the raw data has no listing identifier),
validate the result, and export it to Excel.

### Decisions
- No missing values were found in Bronze, so no imputation is needed.
- Zeros are valid values (binary flags, studios).
- Extreme prices are flagged by default instead of deleted.

## Step 1: Load the Merged Dataset

In [1]:
import os
import pandas as pd

In [2]:
default_silver = "../data/silver"
silver_path = input(f"Enter silver folder path [{default_silver}]: ").strip()
if silver_path == "":
    silver_path = default_silver

master = pd.read_csv(os.path.join(silver_path, "airbnb_merged.csv"))
print("Loaded shape:", master.shape)

Loaded shape: (51707, 21)


## Step 2: Standardize Columns and Data Types
Rename `realSum` to snake_case, trim text columns, and cast count columns to integers.

In [3]:
master = master.rename(columns={"realSum": "real_sum"})

# Trim whitespace in text columns
text_cols = ["room_type", "city", "day_type"]
i = 0
while i < len(text_cols):
    master[text_cols[i]] = master[text_cols[i]].str.strip()
    i += 1

# Count columns should be integers
master["person_capacity"] = master["person_capacity"].astype(int)
master["bedrooms"] = master["bedrooms"].astype(int)

master.dtypes

real_sum                      float64
room_type                         str
room_shared                      bool
room_private                     bool
person_capacity                 int64
host_is_superhost                bool
multi                           int64
biz                             int64
cleanliness_rating            float64
guest_satisfaction_overall    float64
bedrooms                        int64
dist                          float64
metro_dist                    float64
attr_index                    float64
attr_index_norm               float64
rest_index                    float64
rest_index_norm               float64
lng                           float64
lat                           float64
city                              str
day_type                          str
dtype: object

## Step 3: Derived Columns
- `is_studio`: entire place with zero bedrooms
- `price_per_person`: total price divided by person capacity

In [4]:
master["is_studio"] = (master["bedrooms"] == 0) & (master["room_type"] == "Entire home/apt")
master["price_per_person"] = (master["real_sum"] / master["person_capacity"]).round(2)

print("Studios:", int(master["is_studio"].sum()))

Studios: 4026


## Step 4: Outlier Flags
- `is_iqr_outlier`: price above Q3 + 1.5 * IQR inside its own city
- `is_extreme_price`: price above a user-defined threshold

Extreme prices can be flagged (default) or removed.

In [5]:
# Flag outliers inside each city
master["is_iqr_outlier"] = False
cities = sorted(master["city"].unique())

i = 0
while i < len(cities):
    mask = master["city"] == cities[i]
    prices = master.loc[mask, "real_sum"]
    q1 = prices.quantile(0.25)
    q3 = prices.quantile(0.75)
    upper_bound = q3 + 1.5 * (q3 - q1)
    master.loc[mask, "is_iqr_outlier"] = prices > upper_bound
    i += 1

# Ask the user for the extreme price rule
threshold_input = input("Extreme price threshold in EUR [5000]: ").strip()
if threshold_input == "":
    extreme_threshold = 5000.0
else:
    extreme_threshold = float(threshold_input)

# Keep asking until the mode is valid
mode = ""
while mode != "flag" and mode != "remove":
    mode = input("Extreme prices: flag or remove? [flag]: ").strip().lower()
    if mode == "":
        mode = "flag"

master["is_extreme_price"] = master["real_sum"] > extreme_threshold
extreme_count = int(master["is_extreme_price"].sum())

if mode == "remove":
    master = master[~master["is_extreme_price"]].reset_index(drop=True)
    print(f"Removed {extreme_count} extreme listings")
else:
    print(f"Flagged {extreme_count} extreme listings")

print("IQR outliers:", int(master["is_iqr_outlier"].sum()))

Flagged 26 extreme listings
IQR outliers: 2706


## Step 5: Surrogate Key
The raw data has no listing identifier, so `listing_id` is created.
It will become the primary key of the `Listings` table in the Gold layer.

In [6]:
master.insert(0, "listing_id", range(1, len(master) + 1))
print("First id:", master["listing_id"].min(), "| Last id:", master["listing_id"].max())

First id: 1 | Last id: 51707


## Step 6: Validation
Final checks before export: nulls, duplicates, unique key, and valid values.

In [7]:
print("Shape:", master.shape)
print("Nulls:", int(master.isnull().sum().sum()))
print("Duplicated listing_id:", int(master["listing_id"].duplicated().sum()))
print("Min price:", round(master["real_sum"].min(), 2))
print("Cities:", master["city"].nunique(), "| Day types:", master["day_type"].nunique())
print("Room types:", list(master["room_type"].unique()))
master.head()

Shape: (51707, 26)
Nulls: 0
Duplicated listing_id: 0
Min price: 34.78
Cities: 10 | Day types: 2
Room types: ['Private room', 'Entire home/apt', 'Shared room']


,listing_id,real_sum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,rest_index,rest_index_norm,lng,lat,city,day_type,is_studio,price_per_person,is_iqr_outlier,is_extreme_price
0,1,194.033698,Private room,False,True,2,False,1,0,10.0,...,98.253896,6.846473,4.90569,52.41772,Amsterdam,weekday,False,97.02,False,False
1,2,344.245776,Private room,False,True,4,False,0,0,8.0,...,837.280757,58.342928,4.90005,52.37432,Amsterdam,weekday,False,86.06,False,False
2,3,264.101422,Private room,False,True,2,False,0,1,9.0,...,95.386955,6.646700,4.97512,52.36103,Amsterdam,weekday,False,132.05,False,False
3,4,433.529398,Private room,False,True,4,False,0,1,9.0,...,875.033098,60.973565,4.89417,52.37663,Amsterdam,weekday,False,108.38,False,False
4,5,485.552926,Private room,False,True,2,True,0,0,10.0,...,815.305740,56.811677,4.90051,52.37508,Amsterdam,weekday,False,242.78,False,False


## Step 7: Export to Excel
Save the cleaned dataset as the final Silver output.

In [10]:
output_file = os.path.join(silver_path, "airbnb_clean.xlsx")
master.to_excel(output_file, index=False)
print("Saved:", output_file)

# Confirm the file opens and has the same shape
check = pd.read_excel(output_file)
print("Re-read shape:", check.shape)

Saved: ../data/silver\airbnb_clean.xlsx
Re-read shape: (51707, 26)
